### ACS Age Demographics by Commuter Zone

Processes ACS microdata to produce a summary of population age shares by commuter zone.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

acs_dir = Path("/home/brian/Documents/ACS")
year = 2024

In [2]:
# Match PUMAs to commuter zones
cz_match = pd.read_csv(acs_dir / "data/puma2020_cz1990.csv")
cz_dict = {cz: [(puma, afactor)
                for puma, z, afactor
                in cz_match[cz_match['zone'] == cz].values]
           for cz in cz_match['zone'].unique()}

In [3]:
# Read and pre-process ACS microdata
cols = ['STATE', 'PUMA', 'PWGTP', 'AGEP']
dtypes = {'STATE': str, 'PUMA': str}
puma = lambda x: x['STATE'] + x['PUMA']
a65 = lambda x: np.where(x.AGEP >= 65, 1, 0)
u18 = lambda x: np.where(x.AGEP < 18, 1, 0)
yr = str(year)[2:]
files = [f'data/psam_pusa_{yr}.csv', f'data/psam_pusb_{yr}.csv']

df = pd.concat(
    [(pd.read_csv(acs_dir / file, usecols=cols, dtype=dtypes)
        .query('PWGTP > 0')
        .assign(PUMA = puma,
                A65 = a65,
                U18 = u18))
        [['PWGTP', 'PUMA', 'A65', 'U18']]
    for file in files]).astype({'PUMA': int})

In [4]:
# Calculate age shares by commuter zone
d = {}
for cz, puma_list in cz_dict.items():
    pop = 0
    u18pop = 0
    a65pop = 0
    for puma, afactor in puma_list:
        data = (df[df['PUMA'] == puma]
                  .assign(WGT = lambda x: x.PWGTP * afactor))
        pop += data.WGT.sum()
        u18pop += data.loc[data.U18 == 1, 'WGT'].sum()
        a65pop += data.loc[data.A65 == 1, 'WGT'].sum()

    u18sh = u18pop / pop
    o64sh = a65pop / pop
    results = {'Total': pop, 'Age 0-17': u18sh, 'Age 65+': o64sh}
    d[cz] = results

result = pd.DataFrame(d).T
result.index.name = "CZ90"
result.round(4).to_csv('acs_cz_age.csv', index_label='CZ90')
print(f'Year: {year}, CZs: {len(result)}')
result.head()

Year: 2024, CZs: 741


,Total,Age 0-17,Age 65+
CZ90,,,
6200.0,251182.9861,0.202831,0.206720
6000.0,738932.1897,0.217954,0.164707
6100.0,333842.9753,0.232431,0.192496
6600.0,522770.5918,0.223553,0.172954
9600.0,589988.4728,0.222320,0.173549
